# 01 影像组学对照

这本 Notebook 建立两组对照模型：影像组学，以及影像组学联合临床五变量。先从三种 MRI 的病灶区域提取形状、灰度和纹理特征，再通过特征筛选和逻辑回归预测 PSCI。

输入使用 00 保存的患者清单和完整三维病灶标注，开发折号与 ResNet 相同。确认标注覆盖完整体积后，在配置中将 `annotation_scope` 设为 `full_volume`；运行环境需要准备好 PyRadiomics 及其依赖。

下面先定位配置文件，再定义特征提取和建模函数。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg


def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics", "explain": root / "explain", "segmentation": root / "segmentation",
        "reports": root / "reports",
    }


def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")


def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result


def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "original_resnet_integration_no_architecture_optimization", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)


def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table


def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)


def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)


def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")
    if manifest["notebook_definitions_sha256"] != source_snapshot(cfg):
        raise ValueError("Notebook 定义与数据包导出时不同，请使用对应版本或重新导出。")


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本数据包只有分类缓存，没有原分辨率影像；影像组学和分割需另外准备原生数据。")


In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。

后面的指标计算还会用到 scikit-learn 的 ROC、混淆矩阵和重采样工具。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


In [ ]:
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import resample


## 从预测概率计算分类指标

`threshold_from_development` 在开发队列的 ROC 曲线上寻找 Youden 指数最高的位置，得到把概率转成阳性或阴性的阈值。`classification_metrics` 使用传入的阈值计算准确率、敏感度、特异度和混淆矩阵，同时计算 AUC 与 Brier 分数。

需要置信区间时，函数按结局分层重复抽样，估计 AUC 的 95% 区间。阈值在开发阶段确定，保存后用于时间验证。


In [ ]:
from __future__ import annotations

def threshold_from_development(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability)
    return float(thresholds[np.argmax(tpr - fpr)])


def classification_metrics(y, probability, threshold, n_bootstrap=1000):
    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)
    if len(y) != len(probability) or set(y) != {0, 1} or not np.isfinite(probability).all():
        raise ValueError("标签需包含两类，概率必须有限且与患者一一对应。")
    if ((probability < 0) | (probability > 1)).any():
        raise ValueError("概率不在[0,1]。")
    tn, fp, fn, tp = confusion_matrix(y, probability >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "auc": float(roc_auc_score(y, probability)), "threshold": float(threshold),
              "accuracy": float((tp + tn) / len(y)), "sensitivity": float(tp / (tp + fn)),
              "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(y, probability)),
              "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    if n_bootstrap:
        values = []
        for seed in range(n_bootstrap):
            idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
            values.append(roc_auc_score(y[idx], probability[idx]))
        result.update(auc_ci_low=float(np.percentile(values, 2.5)), auc_ci_high=float(np.percentile(values, 97.5)))
    return result


## 特征提取、筛选与建模

`extract_features` 按患者读取三种 MRI 和人工掩模，分别提取标签 1、2、3 对应区域的特征。形状特征描述同一套掩模的几何形态，因此只保留 T1-FLAIR 下的一套；灰度与纹理特征则分别保留三个序列的结果。

`_search` 定义建模顺序：缺失值填补 → 去掉常量特征 → 标准化 → 单变量特征筛选 → 逻辑回归。内层五折同时选择特征保留比例和正则化参数 C；这些处理都在各次训练数据内拟合。加入临床变量时，五个临床指标单独填补和标准化，再与筛选后的影像组学特征合并。

`fit_radiomics_models` 按固定的开发折完成外层评估，保存每位患者的折外预测、每折入选特征，以及用全部开发患者拟合的最终模型。折外预测（OOF）指患者由未将其用于拟合的外层模型产生的预测。`_selected_features` 用来取回入选特征名称，`predict_radiomics_validation` 则负责用保存好的模型预测时间验证患者，在 06 中使用。


In [ ]:
"""New radiomics baseline, independent of the unchanged ResNet implementation."""
from __future__ import annotations

import json
import numpy as np
import pandas as pd



def extract_features(cfg):
    require_native_images(cfg)
    import SimpleITK as sitk
    import radiomics
    from radiomics import featureextractor
    if cfg["annotation_scope"] != "full_volume":
        raise ValueError("本版三维影像组学要求先确认annotation_scope=full_volume；部分标注需另定义有效ROI范围。")
    p, cohort = paths(cfg), read_cohort(cfg)
    if "annotation_scope" not in cohort or not cohort.annotation_scope.eq("full_volume").all():
        raise ValueError("患者清单中存在局部标注或未确认的病例，不能作为完整三维 ROI 提取特征。")
    if ("physical_geometry_needs_review" not in cohort
            or cohort.physical_geometry_needs_review.astype(str).str.lower().ne("false").any()):
        raise ValueError("患者空间信息尚未全部核实，请先处理00记录的单位和间距问题，再提取三维物理特征。")
    destination = p["radiomics"] / "features.csv"
    fresh(destination)
    extractor = featureextractor.RadiomicsFeatureExtractor(**cfg["radiomics"]["settings"])
    extractor.disableAllFeatures()
    for feature_class in cfg["radiomics"]["feature_classes"]:
        extractor.enableFeatureClassByName(feature_class)
    p["radiomics"].mkdir(parents=True, exist_ok=True)
    records, failures = [], []
    for _, patient in cohort.iterrows():
        row = {"mr": patient.mr, "label": int(patient.label), "split": patient.split}
        mask = sitk.ReadImage(patient.manual_mask)
        values = set(np.unique(sitk.GetArrayViewFromImage(mask)))
        for label in (1, 2, 3):
            row[f"roi_{label}_present"] = float(label in values)
            if label not in values:
                continue
            for sequence in SEQUENCES:
                try:
                    features = extractor.execute(str(patient[sequence]), mask, label=label)
                    for name, value in features.items():
                        if not name.startswith("original_") or ("_shape_" in name and sequence != "t1"):
                            continue
                        value = float(value)
                        row[f"{sequence}__roi{label}__{name}"] = value if np.isfinite(value) else np.nan
                except (ValueError, RuntimeError) as exc:
                    failures.append({"mr": patient.mr, "sequence": sequence, "label": label, "error": str(exc)})
        records.append(row)
    pd.DataFrame(failures, columns=["mr", "sequence", "label", "error"]).to_csv(p["radiomics"] / "extraction_errors.csv", index=False)
    if failures:
        raise ValueError("特征提取存在失败；已记录extraction_errors.csv，不发布不完整特征表。")
    table = pd.DataFrame(records)
    table.to_csv(destination, index=False)
    dump_json(p["radiomics"] / "extraction_config.json", {
        "parameters": cfg["radiomics"], "pyradiomics_version": radiomics.__version__,
        "roi_scope": "full_volume_native_geometry", "cohort_sha256": digest(p["cohort"]),
        "note": "固定逐病例提取；不使用验证患者的群体统计。全三维ROI与四层DL覆盖范围不同。"})
    record(cfg, "radiomics_features")
    return table


def _search(cfg, radiomic_columns, clinical_columns):
    from sklearn.compose import ColumnTransformer
    from sklearn.feature_selection import SelectPercentile, VarianceThreshold, f_classif
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    rad = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                    ("variance", VarianceThreshold()), ("scale", StandardScaler()),
                    ("select", SelectPercentile(f_classif))])
    transformers = [("rad", rad, radiomic_columns)]
    if clinical_columns:
        clinical = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)), ("scale", StandardScaler())])
        transformers.append(("clinical", clinical, clinical_columns))
    pipe = Pipeline([("features", ColumnTransformer(transformers)),
                     ("classifier", LogisticRegression(solver="liblinear", max_iter=3000))])
    return GridSearchCV(pipe, {
        "features__rad__select__percentile": cfg["radiomics"]["percentiles"],
        "classifier__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]},
        scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1, error_score="raise")


def _selected_features(model, columns):
    rad = model.named_steps["features"].named_transformers_["rad"]
    retained = np.asarray(columns)[rad.named_steps["variance"].get_support()]
    return retained[rad.named_steps["select"].get_support()].tolist()


def fit_radiomics_models(cfg):
    import joblib
    p, cohort = paths(cfg), read_cohort(cfg)
    fresh(p["radiomics"] / "models.joblib", p["radiomics"] / "model_meta.json")
    extraction = json.loads((p["radiomics"] / "extraction_config.json").read_text(encoding="utf-8"))
    if extraction["cohort_sha256"] != digest(p["cohort"]):
        raise ValueError("提取特征后病例清单发生变化。")
    features = pd.read_csv(p["radiomics"] / "features.csv", dtype={"mr": str})
    if not features.mr.is_unique or set(features.mr) != set(cohort.mr):
        raise ValueError("影像组学特征与固定队列不一一对应。")
    table = cohort.merge(features.drop(columns=["label", "split"]), on="mr", validate="one_to_one")
    table = table.query("split == 'development'").reset_index(drop=True)
    lookup = features.set_index("mr")
    if not np.array_equal(lookup.loc[table.mr, "label"].to_numpy(), table.label.to_numpy()):
        raise ValueError("影像组学结局与队列不一致。")
    radiomic_columns = [c for c in features if c not in ("mr", "label", "split")]
    if not radiomic_columns:
        raise ValueError("没有影像组学特征。")
    y = table.label.to_numpy(dtype=int)
    fitted, metadata, predictions, selection_rows = {}, {}, [], []
    for name, clinical_columns in (("Radiomics", []), ("Radiomics_Clinical5", CLINICAL)):
        columns = radiomic_columns + clinical_columns
        X = table[columns].replace([np.inf, -np.inf], np.nan)
        oof, fold_auc = np.zeros(len(y), dtype=float), []
        for fold in range(1, 6):
            valid = table.development_fold.to_numpy() == fold
            search = _search(cfg, radiomic_columns, clinical_columns)
            search.fit(X.loc[~valid], y[~valid])
            oof[valid] = search.predict_proba(X.loc[valid])[:, 1]
            from sklearn.metrics import roc_auc_score
            fold_auc.append(float(roc_auc_score(y[valid], oof[valid])))
            for feature in _selected_features(search.best_estimator_, radiomic_columns):
                selection_rows.append({"model": name, "fold": fold, "feature": feature})
        search = _search(cfg, radiomic_columns, clinical_columns)
        search.fit(X, y)
        fitted[name] = search.best_estimator_
        threshold = threshold_from_development(y, oof)
        if not np.isfinite(threshold):
            raise ValueError(f"{name}未形成有限Youden阈值；先检查开发预测。")
        metadata[name] = {"features": columns, "threshold": threshold, "best_params": search.best_params_,
                          "selected_features": _selected_features(search.best_estimator_, radiomic_columns),
                          "mean_fold_auc": float(np.mean(fold_auc)), "fold_aucs": fold_auc,
                          "development": classification_metrics(y, oof, threshold, n_bootstrap=0)}
        prediction = table[["mr", "label", "development_fold"]].copy()
        prediction["model"], prediction["probability"] = name, oof
        predictions.append(prediction)
    joblib.dump(fitted, p["radiomics"] / "models.joblib")
    pd.concat(predictions).to_csv(p["radiomics"] / "development_oof.csv", index=False)
    pd.DataFrame(selection_rows).to_csv(p["radiomics"] / "selected_features_by_fold.csv", index=False)
    dump_json(p["radiomics"] / "model_meta.json", metadata)
    dump_json(p["radiomics"] / "frozen_artifacts.json", {"artifacts": {
        filename: digest(p["radiomics"] / filename) for filename in ("models.joblib", "model_meta.json", "features.csv", "extraction_config.json")},
        "cohort_sha256": digest(p["cohort"])})
    record(cfg, "radiomics_models", {"feature_selection": "inside_inner_cv"})
    return metadata


def predict_radiomics_validation(cfg):
    import joblib
    p, cohort = paths(cfg), read_cohort(cfg)
    lock = json.loads((p["radiomics"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if lock["cohort_sha256"] != digest(p["cohort"]):
        raise ValueError("固定队列发生变化。")
    for name, expected in lock["artifacts"].items():
        if digest(p["radiomics"] / name) != expected:
            raise ValueError(f"影像组学固定产物发生变化：{name}")
    feature_table = pd.read_csv(p["radiomics"] / "features.csv", dtype={"mr": str})
    val = cohort.query("split == 'validation'").merge(feature_table.drop(columns=["label", "split"]), on="mr", validate="one_to_one")
    models = joblib.load(p["radiomics"] / "models.joblib")
    meta = json.loads((p["radiomics"] / "model_meta.json").read_text(encoding="utf-8"))
    predictions, summaries = [], []
    for name, model in models.items():
        prob = model.predict_proba(val[meta[name]["features"]].replace([np.inf, -np.inf], np.nan))[:, 1]
        threshold = meta[name]["threshold"]
        table = val[["mr", "label"]].copy()
        table["model"], table["probability"], table["threshold"] = name, prob, threshold
        table["prediction"] = (prob >= threshold).astype(int)
        table["protocol"] = "radiomics_full_volume_nested_feature_selection"
        predictions.append(table)
        summaries.append({"model": name, **classification_metrics(val.label, prob, threshold)})
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(summaries)


## 1. 查看标注范围和提取参数

先确认标注范围、患者分组和提取配置。当前配置启用强度归一化、`normalizeScale=100`、`binWidth=25`，提取形状、一阶统计及多类纹理特征；这些参数会与特征表一起保存，便于复现实验。


In [ ]:
print("标注范围：", cfg["annotation_scope"])
print("提取配置：", cfg["radiomics"])
if p["cohort"].exists():
    display(read_cohort(cfg).groupby("split").size().rename("n"))


## 2. 提取每位患者的影像组学特征

运行后得到 `features.csv`，每行对应一位患者，字段名包含序列、病灶标签和特征名称。某类病灶不存在时，用 `roi_*_present` 记录，相关特征留为空值，后续在训练折内处理。

若区域存在但提取失败，错误会写入 `extraction_errors.csv`，用于定位具体的患者、序列和标签。先检查这些记录，再继续建模。

这里可以按固定参数提取全队列特征；涉及多位患者统计量的填补、标准化和特征筛选，留到下面的训练步骤完成。


In [ ]:
RUN_EXTRACTION = False

if RUN_EXTRACTION:
    features = extract_features(cfg)
    display(features.head())
    print("患者和字段数：", features.shape)


## 3. 训练两组对照模型

外层五折使用 00 保存的折号，内层五折在每个外层训练集内选择参数。分别完成“影像组学”和“影像组学＋临床”后，再用全部开发患者拟合最终处理流程和分类器。

运行后查看两组的开发指标。模型、开发折外预测和分类阈值会一起保存，供 06 直接加载。


In [ ]:
RUN_RADIOMICS_FIT = False

if RUN_RADIOMICS_FIT:
    radiomics_metadata = fit_radiomics_models(cfg)
    display({name: value["development"] for name, value in radiomics_metadata.items()})


## 4. 查看开发结果与特征入选情况

结果表同时列出各折 AUC 的平均值和全部折外预测合并后的 AUC。特征入选频率统计每个特征在多少个外层折中被选中，可以帮助观察筛选结果是否稳定；入选次数本身不等于预测贡献的大小。


In [ ]:
import json
import pandas as pd
meta_path = p["radiomics"] / "model_meta.json"
if meta_path.exists():
    metadata = json.loads(meta_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([{ "model": name, "mean_fold_auc": value["mean_fold_auc"], **value["development"]}
                          for name, value in metadata.items()]))
selected_path = p["radiomics"] / "selected_features_by_fold.csv"
if selected_path.exists():
    selected = pd.read_csv(selected_path)
    display(selected.groupby(["model", "feature"]).size().rename("fold_count").sort_values(ascending=False).head(30))


## 结果文件与比较口径

`radiomics/` 中保存特征表、每折入选特征、开发折外预测、最终模型及阈值。接下来到 06，和 ResNet、临床模型一起完成时间验证。

解释组间差异时，要同时考虑输入范围：这里使用完整三维病灶，ResNet 使用选定的四层影像。两组性能差异可能同时受到建模方法和影像覆盖范围影响。

参数含义可查阅 [PyRadiomics 配置说明](https://pyradiomics.readthedocs.io/en/latest/customization.html)，特征定义可参考 [IBSI](https://theibsi.github.io/)。
